# Ultrasound scan

thicknesses on https://app.notion.com/p/chang-lab/5d544a4e408a48a3bf97c23bf7911c92?v=6a47709549874d7b83bbeaf16146c1f4

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..') # path to the src directory
sys.path.append('/Users/xz498/Library/CloudStorage/OneDrive-DrexelUniversity/Chang Lab - Documents/General/Individual/Xinqiao Zhang/data_analysis/ultrasonicTesting/')

import pickleJar as pj
import sqliteUtils as squ
import numpy as np
from numpy import fft
import os.path
import matplotlib.pyplot as plt

import ipywidgets

In [ ]:
# First we load the data from the file we want to analyze
# The experiment should always output sqlite3 files, so let's convert them to a more usable pickle form first

sqliteFile = '/Users/xz498/Library/CloudStorage/OneDrive-DrexelUniversity/Chang Lab - Documents/General/Individual/Xinqiao Zhang/tomography_toy_data/LMFP_Cu_full_pouch_open.sqlite3'
# Convert the sqlite3 to a .pickle
# This takes a few seconds. A progress bar will display in your terminal
pj.sqliteToPickle(sqliteFile)

# Load the pickle
pickleFile = os.path.splitext(sqliteFile)[0] + '.pickle'

data = pj.loadPickle("/Users/xz498/Library/CloudStorage/OneDrive-DrexelUniversity/Chang Lab - Documents/General/Individual/Xinqiao Zhang/tomography_toy_data/LMFP_Cu_full_pouch_open.pickle")
data['parameters']['dt'] = data['parameters']['measureTime']-data['parameters']['measureDelay'] *10-6/ data['parameters']['samples']

sqliteToPickle Warning: pickle file /Users/xz498/Library/CloudStorage/OneDrive-DrexelUniversity/Chang Lab - Documents/General/Individual/Xinqiao Zhang/temperature testing/LMFP_Cu_gradient.pickle already exists. Conversion aborted.


data['parameters].keys()

In [ ]:
x,z,t = 0,0,0
idx = z*data['parameters']['primaryAxisRange'] + x #+1?
scan_type = 'voltage_echo_reverse'


fig, ax = plt.subplots(2,4 figsize=(10, 8))

img = np.array([dat[scan_type] for k,dat in data.items() if k.isdigit()]).reshape((data['parameters']['primaryAxisRange']*data['parameters']['secondaryAxisRange']))
for i,scantype in enumerate(['voltage_transmission_forward','voltage_transmission_reverse','voltage_echo_reverse','voltage_echo_forward']):
    ax[0,i].imshow(img[...,t])
    ax[0,i].scatter(x,z,color='red',s=5)
    ax[0,i].set_title(f'{scantype} Image at t={data[idx]["Time"][t]} ns')
    ax[0,i].set_xlabel('X (mm)')
    ax[0,i].set_ylabel('Z (mm)') 
    
    ax[1,i].plot(data[idx]['Time'], data[idx][scan_type])
    ax[1,i].plot_vline(data[idx]['Time'][t], color='red', linestyle='--')
    ax[1,i].set_title(f'{scantype} signal at ({x},{z})mm')
    ax[1,i].set_xlabel('Time (ns)')
    ax[1,i].set_ylabel('Voltage (mV)')

# PyBaMM

In [ ]:
# %load_ext autoreload
# % autoreload 2

import pybamm

In [ ]:
options = {"thermal": "x-full", 
           'cell geometry': 'pouch',
           'surface temperature': 'lumped'
           } 
# https://docs.pybamm.org/en/stable/source/api/models/base_models/base_battery_model.html#pybamm.BatteryModelOptions

model = pybamm.lithium_ion.SPMe(options)

In [ ]:
parameter_values = pybamm.ParameterValues("Marquis2019") 
# https://iopscience.iop.org/article/10.1149/2.0341915jes
# https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/parameter-values.html
# TODO: look through the datasheet of the pouch cell.
